In [1]:
import sys

sys.path.append("../")

from langchain_core.stores import InMemoryStore

from langchain_classic.retrievers import ParentDocumentRetriever

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter
)

from src.loaders.document_loader import EnterpriseDocumentLoader
from src.embeddings.embedding_manager import EmbeddingManager
from vector_store.faiss_store import EnterpriseFAISS

c:\Users\gagan\Desktop\Langchain\labs\myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
loader = EnterpriseDocumentLoader()

documents = loader.load_directory(
    "../datasets/military"
)

print(len(documents))

18


In [3]:
parent_splitter = RecursiveCharacterTextSplitter(

    chunk_size=2000,

    chunk_overlap=200
)

In [4]:
child_splitter = RecursiveCharacterTextSplitter(

    chunk_size=400,

    chunk_overlap=80
)

In [5]:
embedding = EmbeddingManager(

    provider="ollama",

    model_name="nomic-embed-text"
)

In [6]:
from langchain_community.vectorstores import FAISS

if documents:
    vectorstore = FAISS.from_documents(
        documents,
        embedding.model
    )
else:
    print("No documents found.")

In [7]:
store = InMemoryStore()

In [8]:
retriever = ParentDocumentRetriever(

    vectorstore=vectorstore,

    docstore=store,

    child_splitter=child_splitter,

    parent_splitter=parent_splitter,
)

In [9]:
retriever.add_documents(
    documents
)

In [10]:
docs = retriever.invoke(
    "Radar Calibration Procedure"
)

len(docs)

1

In [11]:
for doc in docs:

    print("="*80)

    print(doc.metadata)

    print()

    print(doc.page_content[:1000])

{'producer': 'Skia/PDF m151 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Radar_Manual', 'source': '..\\datasets\\military\\Radar_Manual.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'filename': 'Radar_Manual.pdf', 'extension': '.pdf', 'domain': 'military'}

===
  SECTION  1:  SYSTEM  OVERVIEW  ---------------------------  The  XR-7  "Sky  Eye"  provides  real-time  aerial  surveillance  with  a  sweep   radius  of  450  meters.  Enemy  positions  are  relayed  to  all  friendly  HUD   systems  via  encrypted  TAC-LINK  channels.   SWEEP  PATTERNS:  -  Standard  Sweep:  6-second  interval,  360°  coverage  -  Focused  Sweep:  2-second  interval,  120°  directional  cone  -  Overwatch  Mode:  Stationary  90°  persistent  scan   SECTION  2:  THREAT  CLASSIFICATION  ---------------------------------  ICON  LEGEND:  [▲]  -  Hostile  Infantry  (Red  Diamond)  [▲▲]  -  Armored  Vehicle  (Red  Double  Diamond)  [■]  -  Enemy  Killstreak  Active  (Flashing  R

In [12]:
from src.retriever.parent_retriever import EnterpriseParentRetriever

parent = EnterpriseParentRetriever(
    embedding
)

parent.add_documents(documents)

docs = parent.search(
    "Radar Calibration"
)

print(len(docs))

2
